# Lab 3 - Part 1: Arabic Football Classification
Run with Kaggle Internet ON and GPU T4 ON.


In [ ]:
import random, re, time, urllib.parse, urllib.robotparser
from pathlib import Path
from datetime import datetime, timezone
import numpy as np, pandas as pd, requests
import torch
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA=Path('/kaggle/working/data') if Path('/kaggle/working').exists() else Path.cwd()/"data"
DATA.mkdir(parents=True,exist_ok=True)
print('Device:',DEVICE,'|',DATA)


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','beautifulsoup4','lxml','nltk','qalsadi','sentence-transformers','optuna','seaborn'],check=False)
import nltk
for r in ('punkt','punkt_tab','stopwords'):
    try: nltk.download(r,quiet=True)
    except Exception: pass


## 1. Scrape football and non-football sections


In [ ]:
from bs4 import BeautifulSoup
headers={'User-Agent':'Mozilla/5.0 (compatible; educational-deep-learning-lab/3.0)'}
sites={'hespress':('https://www.hespress.com',['/sport','/politique','/economie','/culture']),
'aljazeera':('https://www.aljazeera.net',['/sport','/economy','/culture','/politics']),
'bbc':('https://www.bbc.com/arabic',['/sport','/business','/scienceandtech','/world'])}
seen_urls=set(); seen_text=set(); rows=[]
def fetch(url):
    try:
        rp=urllib.robotparser.RobotFileParser(); p=urllib.parse.urlparse(url)
        rp.set_url(p.scheme+'://'+p.netloc+'/robots.txt'); rp.read()
        if not rp.can_fetch(headers['User-Agent'],url): return None
        r=requests.get(url,headers=headers,timeout=15); r.raise_for_status(); return r.text
    except Exception as e:
        print('Request failed:',url,e); return None
for source,(base,sections) in sites.items():
    for sec in sections:
        html=fetch(urllib.parse.urljoin(base,sec))
        if not html: continue
        soup=BeautifulSoup(html,'lxml'); links=[]
        for a in soup.find_all('a',href=True):
            href=urllib.parse.urljoin(base,a['href'])
            if href not in seen_urls and href not in links: links.append(href)
        for url in links[:100]:
            if len(rows)>=1000: break
            seen_urls.add(url); html=fetch(url); time.sleep(random.uniform(1,2.5))
            if not html: continue
            s=BeautifulSoup(html,'lxml'); h=s.find('h1'); title=h.get_text(' ',strip=True) if h else ''
            c=s.find('article') or s.find('main') or s
            text=re.sub(r'\s+',' ',' '.join(p.get_text(' ',strip=True) for p in c.find_all('p'))).strip()
            if not 50<=len(text.split())<=2000 or text in seen_text: continue
            seen_text.add(text)
            rows.append({'url':url,'source':source,'section':sec.strip('/'),'title':title,'text':text,'date_published':'','scraped_at':datetime.now(timezone.utc).isoformat()})
            if len(rows)%100==0:
                pd.DataFrame(rows).to_csv(DATA/'raw_articles.csv',index=False); print('Saved progress:',len(rows))
    if len(rows)>=1000: break
raw=pd.DataFrame(rows)
if raw.empty: raise RuntimeError('No articles collected. Check Kaggle Internet and source availability.')
raw.to_csv(DATA/'raw_articles.csv',index=False)
print('Collected:',len(raw)); print(raw[['source','section','title']].head())


## 2. Text-only relevance score


In [ ]:
from sentence_transformers import SentenceTransformer
diac=re.compile(r'[\u064B-\u065F\u0670]')
def norm(x):
    x=diac.sub('',str(x)).replace('ـ','').replace('أ','ا').replace('إ','ا').replace('آ','ا').replace('ى','ي').replace('ؤ','و').replace('ئ','ي').replace('ة','ه')
    return re.sub(r'(.)\1{2,}',r'\1',x)
keywords=set(norm(x) for x in 'كرة القدم مباراة هدف الدوري لاعب مدرب فريق منتخب بطولة كأس العالم ركلة جزاء حارس المرمى الميركاتو الانتقالات الفوز التعادل الهزيمة حكم ملعب جماهير المهاجم المدافع الأهلي الوداد الرجاء ريال مدريد برشلونة مانشستر ليفربول'.split())
tok=[norm(t).split() for t in raw.text]
dens=[100*sum(t in keywords for t in ts)/len(ts) if ts else 0 for ts in tok]
D=float(np.percentile(dens,95)); kw=np.array([min(1,d/D) if D else 0 for d in dens])
emb_model=SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
topic=['مباراة كرة القدم بين فريقين في الملعب','هدف اللاعب والدوري والكأس','المدرب والمنتخب والبطولة','ركلة جزاء وحارس المرمى والجماهير','كرة القدم واللاعبون والأندية والبطولات','نتيجة المباراة والهداف والتعادل والفوز','دوري أبطال أوروبا والانتقالات','أخبار كرة القدم واللاعبين والمدربين','التصفيات والبطولات القارية والدولية','الدوري والكأس والمنتخب الوطني']
ce=emb_model.encode(raw.text.str[:256].tolist(),normalize_embeddings=True,show_progress_bar=True)
tv=emb_model.encode(topic,normalize_embeddings=True).mean(axis=0)
sim=ce@tv; p5,p95=np.percentile(sim,[5,95]); emb=np.clip((sim-p5)/(p95-p5+1e-9),0,1)
raw['score']=np.round(10*np.clip(.5*kw+.5*emb,0,1),1)
if np.bincount(pd.cut(raw.score,[-np.inf,3.5,7,np.inf],labels=[0,1,2]).astype(int),minlength=3).min()/len(raw)<.15:
    low,high=np.percentile(raw.score,[33,66]); print('Quantile thresholds:',low,high)
else:
    low,high=3.5,7.0; print('Fixed thresholds:',low,high)
raw['label']=pd.cut(raw.score,[-np.inf,low,high,np.inf],labels=[0,1,2]).astype(int)
dataset=raw[['text','score','label']].copy(); dataset.to_csv(DATA/'dataset.csv',index=False)
print(raw.score.describe()); print(dataset.head())


## 3. Preprocess, split, vocabulary


In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.isri import ISRIStemmer
from sklearn.model_selection import train_test_split
from collections import Counter
stem=ISRIStemmer(); stop={norm(w) for w in stopwords.words('arabic')}|{'اقرأ','ايضا','المزيد','المصدر','حقوق','النشر'}
def preprocess(x):
    x=re.sub(r'<[^>]+>|https?\S+|www\.\S+',' ',str(x)); x=re.sub(r'[^\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\s]',' ',norm(x))
    return [stem.stem(t) for t in word_tokenize(x) if len(t)>1 and norm(t) not in stop]
for x in dataset.text.head(2): print('BEFORE:',x[:180],'\nAFTER:', ' '.join(preprocess(x)[:50]))
dataset['tokens']=dataset.text.map(preprocess)

dataset = dataset[dataset.tokens.map(len) > 0].reset_index(drop=True)  

empty_token_rows=int(dataset.tokens.map(len).eq(0).sum())
if empty_token_rows:
    print(f'Removing {empty_token_rows} articles with no tokens after preprocessing.')
dataset=dataset.loc[dataset.tokens.map(len)>0].copy()
if dataset.empty:
    raise ValueError('Preprocessing removed every article; review the text cleaning and stop-word filters.')
train,temp=train_test_split(dataset,test_size=.30,random_state=SEED,stratify=dataset.label)
val,test=train_test_split(temp,test_size=.50,random_state=SEED,stratify=temp.label)
c=Counter(t for ts in train.tokens for t in ts); vocab={'<PAD>':0,'<UNK>':1}
for t,n in c.items():
    if n>=2:vocab[t]=len(vocab)
max_len=min(400,max(200,int(np.percentile(train.tokens.map(len),90))))
def encode(ts):
    ids=[vocab.get(t,1) for t in ts[:max_len]]; length=len(ids)
    return ids+[0]*(max_len-length),length
print('Vocabulary:',len(vocab),'max_len:',max_len,'splits:',len(train),len(val),len(test))


## 4. Dataset and recurrent model


In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from torch.nn.utils.rnn import pack_padded_sequence
from torch.nn.utils import clip_grad_norm_
class ArabicDS(Dataset):
    def __init__(self,df):
        e=df.tokens.map(encode); self.ids=torch.tensor([x[0] for x in e]); self.lengths=torch.tensor([x[1] for x in e]); self.labels=torch.tensor(df.label.values)
    def __len__(self): return len(self.labels)
    def __getitem__(self,i): return self.ids[i],self.lengths[i],self.labels[i]
train_ds,val_ds,test_ds=ArabicDS(train),ArabicDS(val),ArabicDS(test)
class RecurrentClassifier(nn.Module):
    def __init__(self,kind,emb,hidden,layers,dropout,bidir=False):
        super().__init__(); self.kind=kind; self.embedding=nn.Embedding(len(vocab),emb,padding_idx=0)
        cls={'rnn':nn.RNN,'gru':nn.GRU,'lstm':nn.LSTM}[kind]
        self.rnn=cls(emb,hidden,num_layers=layers,bidirectional=bidir,batch_first=True,dropout=dropout if layers>1 else 0)
        self.drop=nn.Dropout(dropout); self.fc=nn.Linear(hidden*(2 if bidir else 1),3); self.bidir=bidir
    def forward(self,x,length):
        e=self.embedding(x); p=pack_padded_sequence(e,length.cpu(),batch_first=True,enforce_sorted=False)
        if self.kind=='lstm': _,h=self.rnn(p); h=h[0]
        else: _,h=self.rnn(p)
        h=torch.cat((h[-2],h[-1]),1) if self.bidir else h[-1]
        return self.fc(self.drop(h))
print(train_ds.ids.shape,val_ds.ids.shape,test_ds.ids.shape)


## 5. Hyperparameter tuning and evaluation


In [ ]:
import time, optuna
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import accuracy_score,precision_recall_fscore_support,classification_report,confusion_matrix
counts=train.label.value_counts().sort_index().values; weights=torch.tensor(len(train)/(3*counts),dtype=torch.float32,device=DEVICE)
def train_model(kind,bidir,cfg,epochs=12,quiet=True):
    m=RecurrentClassifier(kind,cfg['emb'],cfg['hidden'],cfg['layers'],cfg['dropout'],bidir).to(DEVICE)
    tl=DataLoader(train_ds,batch_size=cfg['batch'],shuffle=True); vl=DataLoader(val_ds,batch_size=cfg['batch'])
    opt=torch.optim.Adam(m.parameters(),lr=cfg['lr']); crit=nn.CrossEntropyLoss(weight=weights)
    best=None; bl=float('inf'); bad=0; history={'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[]}; start=time.time()
    for ep in range(epochs):
        m.train(); ls=[]; yp=[]; yy=[]
        for x,l,y in tl:
            x,l,y=x.to(DEVICE),l.to(DEVICE),y.to(DEVICE); opt.zero_grad(); out=m(x,l); loss=crit(out,y); loss.backward(); clip_grad_norm_(m.parameters(),1); opt.step(); ls.append(loss.item()); yp.extend(out.argmax(1).cpu()); yy.extend(y.cpu())
        m.eval(); vls=[]; vp=[]
        with torch.no_grad():
            for x,l,y in vl:
                x,l,y=x.to(DEVICE),l.to(DEVICE),y.to(DEVICE); out=m(x,l); vls.append(crit(out,y).item()); vp.extend(out.argmax(1).cpu())
        va=np.mean(vls); history['train_loss'].append(np.mean(ls)); history['val_loss'].append(va); history['train_acc'].append(accuracy_score(yy,yp)); history['val_acc'].append(accuracy_score(val.label.tolist(),vp))
        if va<bl: bl=va; best={k:v.cpu() for k,v in m.state_dict().items()}; bad=0
        else: bad+=1
        if bad>=5: break
    m.load_state_dict(best); history['time']=time.time()-start
    return m,history
configs={'RNN':('rnn',False),'BiRNN':('rnn',True),'GRU':('gru',False),'LSTM':('lstm',False)}
optuna.logging.set_verbosity(optuna.logging.WARNING); best_cfg={}
for name,(kind,bidir) in configs.items():
    def objective(trial):
        c={'emb':trial.suggest_categorical('emb',[100,200,300]),'hidden':trial.suggest_categorical('hidden',[64,128,256]),'layers':trial.suggest_categorical('layers',[1,2]),'dropout':trial.suggest_categorical('dropout',[.2,.3,.5]),'lr':trial.suggest_float('lr',1e-4,1e-2,log=True),'batch':trial.suggest_categorical('batch',[16,32,64])}
        _,h=train_model(kind,bidir,c,epochs=6); return min(h['val_loss'])
    st=optuna.create_study(direction='minimize',sampler=optuna.samplers.TPESampler(seed=SEED)); st.optimize(objective,n_trials=20,show_progress_bar=True); best_cfg[name]=st.best_params
print(pd.DataFrame(best_cfg).T)
results={}; histories={}
for name,(kind,bidir) in configs.items():
    m,h=train_model(kind,bidir,best_cfg[name],epochs=30,quiet=False); histories[name]=h
    out=DATA.parent/("best_"+name.lower()+".pt"); torch.save(m.state_dict(),out)
    loader=DataLoader(test_ds,batch_size=best_cfg[name]['batch']); m.eval(); yp=[]; yy=[]
    with torch.no_grad():
        for x,l,y in loader: yp.extend(m(x.to(DEVICE),l.to(DEVICE)).argmax(1).cpu()); yy.extend(y)
    p,r,f,_=precision_recall_fscore_support(yy,yp,average='macro',zero_division=0)
    results[name]=[accuracy_score(yy,yp),p,r,f,h['time']]
    cm=confusion_matrix(yy,yp); sns.heatmap(cm,annot=True,fmt='d',cmap='Blues'); plt.title(name+' confusion matrix'); plt.show(); print(classification_report(yy,yp,zero_division=0))
print(pd.DataFrame(results,index=['Accuracy','Precision','Recall','F1','Train time (s)']).T)
raw.score.plot.hist(bins=30,title='Football relevance score histogram'); plt.xlabel('Score'); plt.show()
